Let's assume for a minute the Lamarckian theory of evolution. It supports the idea that mutations arise as adaptation to a selective pressure. In the phage-bacterial interaction, this means that exposure of bacteria to the viruses would be the cause of the emergence of phage-resistant bacteria. Without considering the potential mechanism, what would happen if there were N bacteria on an agar plate and each one was exposed to one or more viruses?
If the probability of an acquired resistance mutation were μ<sub>α</sub>, can we calculate the probability of observing *m* mutant bacteria? Remember, these bacteria become resistant, survive, and pass the mutation on to their daughter cells.
If, for instance, the mutation rate were μ<sub>α</sub>=10<sup>-8</sup>, then the probability would formally be:

$$
p(m|N, μ_α)=\frac{N!}{(N-m)!m!} · μ_α^m · (1-μ_α)^{(N-m)}
$$

The first term is the number of ways to choose exactly *m* of *N* individual cells (the permutations). When *m=1*, then this factor would be *N*. This would signify that the mutation could have occurred to any single bacteria in the population. When *m=2*, then this permutation factor is *N(N-1)/2*, which is equal to the number of ways you can choose two unique individuals in a population of size *N*.

The middle term is the probability that a mutation with rate *μ<sub>α</sub>* would occur precisely *m* times.

The last term denotes the sensitivity. It is the probability that a mutation does not occur precisely *N-m* times with the probability *1-μ<sub>α</sub>*.

This binomial distribution is challenging to calculate, even computationally, due to the massive factorials in the permutation term. Given the constraints of the Luria-Delbruck experimental setup, we can make a couple of assumptions:


* N is a very large number
* The mutation rate μ<sub>α</sub> is likely a very small number (albeit, this was unknown when the experiment was conducted).

So, with *N >> 1* and *0 < μ<sub>α</sub> << 1*, the binomial distribution becomes:

$$
p(m|N, μ_α)=\frac{(Nμ_α)^m · e^{-Nμ_α}}{m!}
$$

Looking closely at this formula, it denotes the Poisson distribution and it is the limit of a binomial distribution with a very large number of permutations and very small probability of successful events.

One of the interesting properties of the Poisson distribution is that the variance of it is equal to the mean. This means that the standard deviation (the square root of the variance will be
$$
σ=\bar{m}^{1/2}
$$

In practice, this means that replicate experiments will yield small fluctuations, and the **standard deviation will grow slower than the mean**.

So, assume that you work in the lab and your experiment resulted in the results of Luria and Delbruck, as seen in the table at the top of the previous notebook. You have several plates with zero resistant colonies and a few with hundreds of resistant colonies each.
If mutations depend on selection, we would expect sometimes to see no mutants at all. The mathematical expression of this would be:

$$
p(0|N, μ_α)=\frac{(Nμ_α)^0 · e^{-Nμ_α}}{0!}=e^{-Nμ_α}
$$

If the fraction of replicates with zero colonies is *f<sub>0</sub>*, the best estimate of the mutation rate would be:

$$
\hatμ_α=-\frac{logf_0}{N}
$$

This creates problems when *f<sub>0</sub>*=0, as the mutation rate would be undefined.

A feature of the Poisson distibution is that the average number of observed colonies is predicted to be *Nμ<sub>α</sub>*. In this case, the best estimate for the mutation rate would be:

$$
\hatμ_α=-\frac{\bar{m}_{obs}}{N}
$$

Using the mean as the basis for estimating the mutation rate makes sense, as it is practically the maximum likelihood estimate of the rate. This does not always need to be the case. However, remember that, in the Poisson distibution, the variance is equal to the mean. This means that the standard deviation (which is the square root of the variance) should be smaller than the mean.

If for instance you had on average 8 colonies per plate, you would expect most plates to have somewhere around 5 and 12 colonies. Large deviations would be rare. The table of results from the Luria Delbruck experiment shows that this was not the case.

The data then **do not seem to quantitatively support** a mechanism of mutation that is dependent on selection.

One of the interesting features in the Luria-Delbruck experiment was the fraction of the experiments in which nothing happened. In other words, how many plates showed no growth of mutant colonies? This feature, the probability of zeros, can be used to infer a rate parameter (the estimate of λ). Let's use the same dataset here. The *csv* file

In [ ]:
# Read csv file from Google Drive
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/My Drive/Weitz_pois.csv') # <- Make sure to enter the correct directory path here.
print(df.head())

In [ ]:
# Read data from file:
poissdata=np.genfromtxt('/content/drive/My Drive/Weitz_pois.csv', delimiter=',')
numberofzero = np.sum(df==0)
# Generate the probability of zeros:
probzero=numberofzero/len(poissdata)
print(probzero)

The following cell will create a function that estimates the λ for the Poisson distribution.

In [ ]:
#create lambda estimator function
def lambda_est_zero(x):
  # estimate the poisson rate parameter associated with a vector of points x based on the zero
  numberofzero=np.sum((x==0)*1)
  probzero=numberofzero/len(x)
  lambda_est= -np.log(probzero)
  return lambda_est

In [ ]:
# Run the function by running it for your 'poissdata' dataset.
lambda_est = lambda_est_zero(poissdata)
print(lambda_est)

In the Poisson distribution, if there are three zeros in 100 trials (as in this dataset), the observed probability is P*<sub>obs</sub>*(0)=0.03. The best estimate for λ is *λ<sub>est</sub>=-log(P<sub>obs</sub>)=3.51*. This number is not quire right. Since there are actually three zeros over 100 measurements, *λ<sub>true</sub>=3*. But our estimated λ is good enough. Keep in mind that Luria and Delbruck did not know the mutation rate before the experiment.

How can you be confident of such estimates? This is where statistics steps in. One way to quantify similarity to a predetermined rate is to ask whether λ<sub>est</sub> lies within the middle 95% of the distribution of estimates. The following cell will generate a random poisson distribution, estimate the λ, and plot the normalized histogram.

In [ ]:
nums=10**4
lambdadist=np.zeros(nums)
for i in range(nums):
  currdata = np.random.poisson(lam=lambda_est, size=100)
  lambdadist[i] = np.mean(currdata)

n, bin_edges = np.histogram(lambdadist, bins=30)
bin_prob = n / nums
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
bin_width = bin_edges[1] - bin_edges[0]
plt.bar(bin_centers, bin_prob, width=bin_width)
plt.plot([lambda_est, lambda_est],[0,0.12],'k--', linewidth = 3)
plt.ylabel(r'Probability distribution, $p(\lambda)$', fontsize=15)
plt.xlabel(r'Value of $\lambda$', fontsize=15)

To estimate the confidence intervals of the estimated value of λ, you need to establish the minimum and maximum values of the true λ. You can do this by looping over different λ values and repeating the analysis.

In [ ]:
# Loop over λ values and repeat the analysis to estimate confidence intervals
# takes approx. 2 minutes to run
lambda_vec = np.arange(0.5*lambda_est, 1.5*lambda_est, 0.01)
lower_025 = np.zeros(len(lambda_vec))
upper_975 = np.zeros(len(lambda_vec))

for j in range(len(lambda_vec)):
    curr_lambda = lambda_vec[j]
    lambdadist = np.zeros(nums)
    for k in range(nums):
        currdata = np.random.poisson(lam=curr_lambda, size=100)
        lambdadist[k] = np.mean(currdata)
    sortedlambdadist = np.sort(lambdadist)
    lower_025[j] = sortedlambdadist[int(0.025*nums)]
    upper_975[j] = sortedlambdadist[int(0.975*nums)]

Plot the upper and lower bounds of *λ<sub>obs</sub>* given a range of values for *λ<sub>true</sub>*.

In [ ]:
plt.plot(lambda_vec, lower_025, 'k--', linewidth = 3)
plt.plot(lambda_vec, upper_975, 'k--', linewidth = 3)
plt.plot(lambda_vec, lambda_vec, linewidth = 3)
tmpi = np.where(lower_025 > lambda_est)[0]
limlow = lambda_vec[tmpi[0]]
tmpi = np.where(upper_975 < lambda_est)[0]
limhigh = lambda_vec[tmpi[-1]]
plt.plot([limlow, limhigh], [lambda_est, lambda_est], linewidth = 4)
plt.xlim([1.25,5.75])
plt.ylim([1.25,5.75])
plt.ylabel(r'Confidence Intervals', fontsize=15)
plt.xlabel(r'True parameter $\lambda$', fontsize=15)

You can repeat the analysis using a different summary statistic. This time, instead of the number of zero occurrences, use the mean of the Poisson distribution as an equivalent to λ.

In [ ]:
# Start your new analysis here.
